# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.1 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F

In [5]:
TASK_ID='task133'
H=W=30
FORBIDDEN_OPS={'Loop','Scan','NonZero','Unique','Script','Function'}

# Write/import the constructive model module. This avoids relying on a prebuilt ONNX file.
MODULE_CODE = "import json, os, time, hashlib, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nH=W=30\nPATTERNS = [\n ((0,1),(1,0)), ((0,1),(1,0),(1,1)), ((0,1),(0,2)), ((1,0),(1,1)), ((1,0),(2,0)),\n ((0,1),(0,2),(1,0)), ((0,1),(0,2),(1,0),(1,2)), ((0,1),(1,0),(2,0)), ((0,1),(1,1)),\n ((0,1),(1,0),(2,0),(2,1)), ((0,1),(0,2),(1,0),(1,1)), ((0,1),(0,2),(1,1)),\n ((0,1),(0,2),(1,0),(2,0)), ((1,0),(2,0),(2,1)), ((1,0),(1,1),(2,0)), ((0,1),(1,0),(1,1),(2,0)),\n ((0,1),(0,2),(1,2)), ((1,0),(1,1),(2,0),(2,1)), ((0,1),(1,0),(1,1),(1,2)), ((1,0),(1,1),(1,2)),\n ((1,0),(1,1),(2,1)), ((0,1),(0,2),(1,1),(2,1)), ((0,1),(1,0),(1,1),(2,1)), ((0,1),(0,2),(1,1),(1,2)),\n ((1,0),(1,1),(1,2),(2,0)), ((0,1),(0,2),(1,2),(2,2)), ((0,1),(1,1),(1,2)), ((0,1),(1,1),(2,1)),\n ((1,0),(1,1),(1,2),(2,1)), ((-1,-1),(0,-2),(0,-1),(1,-1)), ((-1,3),(0,1),(0,2),(0,3),(1,3)),\n ((-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,1)), ((-1,0),(0,-2),(0,-1),(1,0)),\n ((-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0)), ((0,1),(1,1),(1,2),(2,2)), ((0,1),(1,1),(2,1),(2,2)),\n ((0,1),(1,1),(1,2),(2,1)), ((0,2),(1,0),(1,1),(1,2)), ((1,0),(1,1),(2,1),(2,2)), ((1,0),(2,0),(2,1),(2,2)),\n]\nP=len(PATTERNS); ALL_OFF=[o for p in PATTERNS for o in p]\nMIN_DY=min(d for d,_ in ALL_OFF); MAX_DY=max(d for d,_ in ALL_OFF); MIN_DX=min(x for _,x in ALL_OFF); MAX_DX=max(x for _,x in ALL_OFF)\nBH=MAX_DY-MIN_DY+1; BW=MAX_DX-MIN_DX+1\nclass Task133DynKernel(nn.Module):\n    def __init__(self):\n        super().__init__()\n        sw=torch.zeros(9*P,1,BH,BW)\n        plen=[]\n        for g in range(9):\n            for pi,pat in enumerate(PATTERNS):\n                for dy,dx in pat: sw[g*P+pi,0,dy-MIN_DY,dx-MIN_DX]=1\n                plen.append(len(pat))\n        self.register_buffer('source_w',sw)\n        self.register_buffer('plen',torch.tensor(plen,dtype=torch.float32).view(1,9,P,1,1))\n        for s in range(1,5):\n            core=torch.ones(1,1,s,s)\n            ring=torch.zeros(1,1,s+2,s+2)\n            ring[:,:,0,1:s+1]=1; ring[:,:,s+1,1:s+1]=1\n            ring[:,:,1:s+1,0]=1; ring[:,:,1:s+1,s+1]=1\n            self.register_buffer(f'core_{s}',core); self.register_buffer(f'ring_{s}',ring)\n            bw=torch.zeros(P,1,BH*s,BW*s)\n            for pi,pat in enumerate(PATTERNS):\n                for dy,dx in pat:\n                    iy=(dy-MIN_DY)*s; ix=(dx-MIN_DX)*s\n                    bw[pi,0,iy:iy+s,ix:ix+s]=1.0\n            self.register_buffer(f'base_w_{s}',bw)\n    def top_left_pad(self,y,s): return F.pad(y,(0,s-1,0,s-1))\n    def exact_square1(self,xc,s):\n        core=getattr(self,f'core_{s}'); ring=getattr(self,f'ring_{s}')\n        cs=F.conv2d(xc,core); rs=F.conv2d(F.pad(xc,(1,1,1,1)),ring)\n        return self.top_left_pad(((cs==float(s*s)) & (rs==0.0)).to(xc.dtype),s)\n    def adj_ring1(self,xc,s):\n        ring=getattr(self,f'ring_{s}'); return self.top_left_pad((F.conv2d(F.pad(xc,(1,1,1,1)),ring)>0.0).to(xc.dtype),s)\n    def forward(self,x):\n        x=x.float(); colors=x[:,1:10]; empty=x[:,0:1]\n        total=colors.sum(1,keepdim=True)\n        exact1=[]; exact={}; adj={}\n        for i in range(9):\n            xc=colors[:,i:i+1]\n            e1=self.exact_square1(xc,1); exact1.append(e1)\n            for s in range(1,5):\n                exact[(i,s)]=e1 if s==1 else self.exact_square1(xc,s)\n                adj[(i,s)]=self.adj_ring1(xc,s)\n        nonmarker=torch.cat([torch.clamp(total-colors[:,i:i+1],0,1) for i in range(9)],1)\n        score=F.conv2d(F.pad(nonmarker,(-MIN_DX,MAX_DX,-MIN_DY,MAX_DY)), self.source_w, groups=9)\n        score=score.view(1,9,P,H,W)  # static batch 1\n        ex1=torch.cat(exact1,1).view(1,9,1,H,W)\n        sp=((score==self.plen)*ex1).to(x.dtype).amax(dim=(3,4)) # [1,9,P]\n        add=torch.zeros_like(colors)\n        for midx in range(9):\n            spm=sp[:,midx,:].view(P,1,1,1)  # static B=1\n            for s in range(1,5):\n                T=[]\n                for cidx in range(9):\n                    t=exact[(midx,s)]*adj[(cidx,s)]\n                    if cidx==midx: t=t*0.0\n                    T.append(t)\n                T=torch.cat(T,1)\n                base=getattr(self,f'base_w_{s}')\n                w=(base*spm).amax(dim=0,keepdim=True) # [1,1,kH,kW], source has one pattern; max safer than sum\n                w=w.repeat(9,1,1,1)\n                y=F.conv_transpose2d(T,w,groups=9)\n                oy=(-MIN_DY)*s; ox=(-MIN_DX)*s\n                y=y[:,:,oy:oy+H,ox:ox+W]\n                add=torch.maximum(add,y)\n        any_add=(add*empty).amax(dim=1,keepdim=True)\n        return torch.cat([empty*(1-any_add), torch.maximum(colors,add*empty)],1)\n\ndef onehot(grid):\n    arr=np.zeros((1,10,H,W),np.float32); g=np.array(grid,dtype=np.int64); h,w=g.shape\n    for y in range(h):\n        for x in range(w): arr[0,g[y,x],y,x]=1\n    return arr\n\ndef eval_model(model,exs):\n    r=n=0\n    with torch.no_grad():\n        for ex in exs:\n            inp=np.array(ex['input']); exp=np.array(ex['output'])\n            pred=model(torch.from_numpy(onehot(inp))).numpy()[0].argmax(0)[:inp.shape[0],:inp.shape[1]]\n            r+=int(np.array_equal(pred,exp)); n+=1\n    return r,n\n\ndef split_arc(arc):\n    tr=[]; ho=[]\n    for ex in arc:\n        g=np.array(ex['input']); h,w=g.shape; colors=tuple(sorted(map(int,set(g.flatten())-{0})))\n        counts=tuple(sorted([int((g==c).sum()) for c in colors]))\n        key=(h//4,w//4,len(colors),tuple(c//3 for c in counts))\n        hv=int(hashlib.sha1(str(key).encode()).hexdigest(),16)%10\n        (ho if hv>=7 else tr).append(ex)\n    return tr,ho\n"
sys.path.insert(0, str(Path.cwd()))


TASK_PATH=Path(COMPETITION)/f'{TASK_ID}.json'
task=json.load(open(TASK_PATH))
print('Loaded', TASK_PATH, {k:len(v) for k,v in task.items()})

Loaded /kaggle/input/competitions/neurogolf-2026/task133.json {'train': 4, 'test': 1, 'arc-gen': 262}


In [6]:
PATTERNS = [
 ((0,1),(1,0)), ((0,1),(1,0),(1,1)), ((0,1),(0,2)), ((1,0),(1,1)), ((1,0),(2,0)),
 ((0,1),(0,2),(1,0)), ((0,1),(0,2),(1,0),(1,2)), ((0,1),(1,0),(2,0)), ((0,1),(1,1)),
 ((0,1),(1,0),(2,0),(2,1)), ((0,1),(0,2),(1,0),(1,1)), ((0,1),(0,2),(1,1)),
 ((0,1),(0,2),(1,0),(2,0)), ((1,0),(2,0),(2,1)), ((1,0),(1,1),(2,0)), ((0,1),(1,0),(1,1),(2,0)),
 ((0,1),(0,2),(1,2)), ((1,0),(1,1),(2,0),(2,1)), ((0,1),(1,0),(1,1),(1,2)), ((1,0),(1,1),(1,2)),
 ((1,0),(1,1),(2,1)), ((0,1),(0,2),(1,1),(2,1)), ((0,1),(1,0),(1,1),(2,1)), ((0,1),(0,2),(1,1),(1,2)),
 ((1,0),(1,1),(1,2),(2,0)), ((0,1),(0,2),(1,2),(2,2)), ((0,1),(1,1),(1,2)), ((0,1),(1,1),(2,1)),
 ((1,0),(1,1),(1,2),(2,1)), ((-1,-1),(0,-2),(0,-1),(1,-1)), ((-1,3),(0,1),(0,2),(0,3),(1,3)),
 ((-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,1)), ((-1,0),(0,-2),(0,-1),(1,0)),
 ((-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0)), ((0,1),(1,1),(1,2),(2,2)), ((0,1),(1,1),(2,1),(2,2)),
 ((0,1),(1,1),(1,2),(2,1)), ((0,2),(1,0),(1,1),(1,2)), ((1,0),(1,1),(2,1),(2,2)), ((1,0),(2,0),(2,1),(2,2)),
]
P=len(PATTERNS); ALL_OFF=[o for p in PATTERNS for o in p]
MIN_DY=min(d for d,_ in ALL_OFF); MAX_DY=max(d for d,_ in ALL_OFF); MIN_DX=min(x for _,x in ALL_OFF); MAX_DX=max(x for _,x in ALL_OFF)
BH=MAX_DY-MIN_DY+1; BW=MAX_DX-MIN_DX+1

In [7]:

def find_task_json():
    candidates=[Path(f'{TASK_ID}.json'), Path('/mnt/data')/f'{TASK_ID}.json']
    candidates += [Path(p) for p in glob.glob(f'/kaggle/input/**/{TASK_ID}.json', recursive=True)]
    for p in candidates:
        if p.exists(): return p
    raise FileNotFoundError(f'Could not find {TASK_ID}.json')



def onehot(grid):
    arr=np.zeros((1,10,H,W),np.float32)
    g=np.array(grid,dtype=np.int64); h,w=g.shape
    for y in range(h):
        for x in range(w): arr[0,g[y,x],y,x]=1.0
    return arr

def structural_split(arc):
    train=[]; hold=[]
    for ex in arc:
        g=np.array(ex['input']); h,w=g.shape
        colors=tuple(sorted(map(int,set(g.flatten())-{0})))
        counts=tuple(sorted([int((g==c).sum()) for c in colors]))
        key=(h//4,w//4,len(colors),tuple(c//3 for c in counts))
        hv=int(hashlib.sha1(str(key).encode()).hexdigest(),16)%10
        (hold if hv>=7 else train).append(ex)
    return train,hold


In [8]:

class Task133DynKernel(nn.Module):
    def __init__(self):
        super().__init__()
        sw=torch.zeros(9*P,1,BH,BW)
        plen=[]
        for g in range(9):
            for pi,pat in enumerate(PATTERNS):
                for dy,dx in pat: sw[g*P+pi,0,dy-MIN_DY,dx-MIN_DX]=1
                plen.append(len(pat))
        self.register_buffer('source_w',sw)
        self.register_buffer('plen',torch.tensor(plen,dtype=torch.float32).view(1,9,P,1,1))
        for s in range(1,5):
            core=torch.ones(1,1,s,s)
            ring=torch.zeros(1,1,s+2,s+2)
            ring[:,:,0,1:s+1]=1; ring[:,:,s+1,1:s+1]=1
            ring[:,:,1:s+1,0]=1; ring[:,:,1:s+1,s+1]=1
            self.register_buffer(f'core_{s}',core); self.register_buffer(f'ring_{s}',ring)
            bw=torch.zeros(P,1,BH*s,BW*s)
            for pi,pat in enumerate(PATTERNS):
                for dy,dx in pat:
                    iy=(dy-MIN_DY)*s; ix=(dx-MIN_DX)*s
                    bw[pi,0,iy:iy+s,ix:ix+s]=1.0
            self.register_buffer(f'base_w_{s}',bw)
    def top_left_pad(self,y,s): return F.pad(y,(0,s-1,0,s-1))
    def exact_square1(self,xc,s):
        core=getattr(self,f'core_{s}'); ring=getattr(self,f'ring_{s}')
        cs=F.conv2d(xc,core); rs=F.conv2d(F.pad(xc,(1,1,1,1)),ring)
        return self.top_left_pad(((cs==float(s*s)) & (rs==0.0)).to(xc.dtype),s)
    def adj_ring1(self,xc,s):
        ring=getattr(self,f'ring_{s}'); return self.top_left_pad((F.conv2d(F.pad(xc,(1,1,1,1)),ring)>0.0).to(xc.dtype),s)
    def forward(self,x):
        x=x.float(); colors=x[:,1:10]; empty=x[:,0:1]
        total=colors.sum(1,keepdim=True)
        exact1=[]; exact={}; adj={}
        for i in range(9):
            xc=colors[:,i:i+1]
            e1=self.exact_square1(xc,1); exact1.append(e1)
            for s in range(1,5):
                exact[(i,s)]=e1 if s==1 else self.exact_square1(xc,s)
                adj[(i,s)]=self.adj_ring1(xc,s)
        nonmarker=torch.cat([torch.clamp(total-colors[:,i:i+1],0,1) for i in range(9)],1)
        score=F.conv2d(F.pad(nonmarker,(-MIN_DX,MAX_DX,-MIN_DY,MAX_DY)), self.source_w, groups=9)
        score=score.view(1,9,P,H,W)  # static batch 1
        ex1=torch.cat(exact1,1).view(1,9,1,H,W)
        sp=((score==self.plen)*ex1).to(x.dtype).amax(dim=(3,4)) # [1,9,P]
        add=torch.zeros_like(colors)
        for midx in range(9):
            spm=sp[:,midx,:].view(P,1,1,1)  # static B=1
            for s in range(1,5):
                T=[]
                for cidx in range(9):
                    t=exact[(midx,s)]*adj[(cidx,s)]
                    if cidx==midx: t=t*0.0
                    T.append(t)
                T=torch.cat(T,1)
                base=getattr(self,f'base_w_{s}')
                w=(base*spm).amax(dim=0,keepdim=True) # [1,1,kH,kW], source has one pattern; max safer than sum
                w=w.repeat(9,1,1,1)
                y=F.conv_transpose2d(T,w,groups=9)
                oy=(-MIN_DY)*s; ox=(-MIN_DX)*s
                y=y[:,:,oy:oy+H,ox:ox+W]
                add=torch.maximum(add,y)
        any_add=(add*empty).amax(dim=1,keepdim=True)
        return torch.cat([empty*(1-any_add), torch.maximum(colors,add*empty)],1)

def onehot(grid):
    arr=np.zeros((1,10,H,W),np.float32); g=np.array(grid,dtype=np.int64); h,w=g.shape
    for y in range(h):
        for x in range(w): arr[0,g[y,x],y,x]=1
    return arr

def eval_model(model,exs):
    r=n=0
    with torch.no_grad():
        for ex in exs:
            inp=np.array(ex['input']); exp=np.array(ex['output'])
            pred=model(torch.from_numpy(onehot(inp))).numpy()[0].argmax(0)[:inp.shape[0],:inp.shape[1]]
            r+=int(np.array_equal(pred,exp)); n+=1
    return r,n

def split_arc(arc):
    tr=[]; ho=[]
    for ex in arc:
        g=np.array(ex['input']); h,w=g.shape; colors=tuple(sorted(map(int,set(g.flatten())-{0})))
        counts=tuple(sorted([int((g==c).sum()) for c in colors]))
        key=(h//4,w//4,len(colors),tuple(c//3 for c in counts))
        hv=int(hashlib.sha1(str(key).encode()).hexdigest(),16)%10
        (ho if hv>=7 else tr).append(ex)
    return tr,ho

In [9]:
OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/task133_v4_compact_out')
OUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=OUT_DIR/f'{TASK_ID}.onnx'

model=Task133DynKernel().eval()
dummy=torch.zeros(1,10,H,W); dummy[:,0]=1.0
torch.onnx.export(model,dummy,str(MODEL_PATH),input_names=['input'],output_names=['output'],opset_version=18,do_constant_folding=True)


[torch.onnx] Obtain model graph for `Task133DynKernel()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Task133DynKernel()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.10.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,10,30,30]>
            ),
            outputs=(
                %"output"<FLOAT,[1,10,30,30]>
            ),
            initializers=(
                %"source_w"<FLOAT,[360,1,4,6]>{TorchTensor(...)},
                %"plen"<FLOAT,[1,9,40,1,1]>{TorchTensor(...)},
                %"core_1"<FLOAT,[1,1,1,1]>{TorchTensor<FLOAT,[1,1,1,1]>(tensor([[[[1.]]]]), name='core_1')},
                %"ring_1"<FLOAT,[1,1,3,3]>{TorchTensor<FLOAT,[1,1,3,3]>(tensor([[[[0., 1., 0.], [1., 0., 1.], [0., 1., 0.]]]]), name='ring_1')},
                %"base_w_1"<FLOAT,[40,1,4,6]>{TorchTensor(...)},
                %"core_2"<FLOAT,[1,1,2,2]>{TorchTensor<FLOAT,[1,1,2,2]>(t

In [10]:
# Force static public I/O metadata [1,10,30,30]. Runtime already returns that tensor shape.
onnx_model=onnx.load(str(MODEL_PATH))
for vi in list(onnx_model.graph.input)+list(onnx_model.graph.output):
    shape=vi.type.tensor_type.shape
    for i,dim in enumerate(shape.dim):
        dim.ClearField('dim_param')
        dim.dim_value=[1,10,30,30][i]
onnx.save(onnx_model,str(MODEL_PATH))
print('Exported',MODEL_PATH,'size',MODEL_PATH.stat().st_size)



Exported /kaggle/working/task133.onnx size 857143


In [11]:
onnx_model=onnx.load(str(MODEL_PATH))
onnx.checker.check_model(onnx_model)
ops=sorted({n.op_type for n in onnx_model.graph.node})
forbidden=sorted(set(ops)&FORBIDDEN_OPS)
assert not forbidden, forbidden
assert MODEL_PATH.stat().st_size < 1_400_000, MODEL_PATH.stat().st_size

so=ort.SessionOptions(); so.graph_optimization_level=ort.GraphOptimizationLevel.ORT_DISABLE_ALL
so.intra_op_num_threads=1; so.inter_op_num_threads=1
session=ort.InferenceSession(str(MODEL_PATH),sess_options=so,providers=['CPUExecutionProvider'])
assert session.get_inputs()[0].shape == [1,10,30,30]
assert session.get_outputs()[0].shape == [1,10,30,30]
print('ONNX I/O',session.get_inputs()[0].shape,session.get_outputs()[0].shape)
print('Opset',[(o.domain,o.version) for o in onnx_model.opset_import])
print('Forbidden ops',forbidden)



ONNX I/O [1, 10, 30, 30] [1, 10, 30, 30]
Opset [('', 18)]
Forbidden ops []


In [12]:
def eval_examples(examples):
    right=0; times=[]
    for ex in examples:
        inp=np.array(ex['input']); exp=np.array(ex['output'])
        t=time.time(); pred=session.run(None,{'input':onehot(inp)})[0][0].argmax(0)[:inp.shape[0],:inp.shape[1]]; times.append(time.time()-t)
        right += int(np.array_equal(pred,exp))
    return {'right':right,'total':len(examples),'avg_runtime_sec':float(np.mean(times)) if times else None,'max_runtime_sec':float(np.max(times)) if times else None}



In [13]:
arc_train,arc_holdout=structural_split(task.get('arc-gen',[]))
report={
    'task_id':TASK_ID,
    'model':'task133_v4_compact_pattern_dynamic_kernel',
    'model_summary':'detect source marker prototype, build dynamic transfer kernel, copy residual prototype to same-marker target blocks at scale 1..4, freeze existing cells',
    'onnx_size_bytes':MODEL_PATH.stat().st_size,
    'public_input_shape':session.get_inputs()[0].shape,
    'public_output_shape':session.get_outputs()[0].shape,
    'opset':[(o.domain,o.version) for o in onnx_model.opset_import],
    'forbidden_ops':forbidden,
    'structural_split':'arc-gen split by grid-size bins, color count, and nonzero-count bins',
    'split_sizes':{'arc_train':len(arc_train),'arc_holdout':len(arc_holdout)},
}


In [14]:
for name,examples in [('visible_train',task.get('train',[])),('visible_test',task.get('test',[])),('arc_train',arc_train),('arc_holdout',arc_holdout),('arc_all',task.get('arc-gen',[]))]:
    report[name]=eval_examples(examples)
    print(name, report[name])
assert report['visible_train']['right']==report['visible_train']['total'], report['visible_train']
assert report['visible_test']['right']==report['visible_test']['total'], report['visible_test']
if report['arc_holdout']['total']:
    assert report['arc_holdout']['right']==report['arc_holdout']['total'], report['arc_holdout']
REPORT_PATH=OUT_DIR/f'{TASK_ID}_v4_compact_validation_report.json'
REPORT_PATH.write_text(json.dumps(report,indent=2))


visible_train {'right': 4, 'total': 4, 'avg_runtime_sec': 0.03460723161697388, 'max_runtime_sec': 0.038953304290771484}
visible_test {'right': 1, 'total': 1, 'avg_runtime_sec': 0.03222203254699707, 'max_runtime_sec': 0.03222203254699707}
arc_train {'right': 173, 'total': 173, 'avg_runtime_sec': 0.03246311231844687, 'max_runtime_sec': 0.036685943603515625}
arc_holdout {'right': 89, 'total': 89, 'avg_runtime_sec': 0.0329053910930505, 'max_runtime_sec': 0.04705071449279785}
arc_all {'right': 262, 'total': 262, 'avg_runtime_sec': 0.032437592062331336, 'max_runtime_sec': 0.03913402557373047}


1367

In [15]:
SUBMISSION_PATH=Path.cwd()/'submission.zip'
with zipfile.ZipFile(SUBMISSION_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH,arcname=f'{TASK_ID}.onnx')
print('Wrote',SUBMISSION_PATH)

Wrote /kaggle/working/submission.zip
